# Notebook 1: Data Ingestion & Exploration
## NYC Yellow Taxi Trip Data Analysis (2024)
### Module: Machine Learning and Big Data (7006SCN)
---

## 1.1 Environment Setup

In [ ]:
# Environment Setup
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["PYSPARK_PYTHON"] = "python"

import findspark
findspark.init()

# Core imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

## 1.2 Spark Session Configuration
Configure SparkSession with optimized settings for Big Data processing.

In [ ]:
# Create SparkSession with optimized configuration
spark = (SparkSession.builder
    .master("local[*]")
    .appName("NYC_Taxi_BigData_Analysis")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "localhost")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate())

# Set log level to reduce verbose output
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Spark Session created successfully!")
print("\nSpark Configuration:")
print(f"  Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"  Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"  Adaptive Query Execution: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"  Serializer: {spark.conf.get('spark.serializer')}")

## 1.3 Data Ingestion
Load NYC Yellow Taxi Trip Records (January - December 2024) from Parquet files.

**Data Source:** NYC Taxi & Limousine Commission via AWS Open Data Registry  
**Format:** Apache Parquet  
**License:** NYC Open Data Terms of Use

In [ ]:
# ============================================================
# IMPORTANT: Update this path to where your parquet files are
# ============================================================
DATA_PATH = r"C:\Users\MaK Tech\Desktop\nyc-taxi-bigdata\data\*.parquet"

# Load all parquet files
print("Loading data...")
df = spark.read.parquet(DATA_PATH)
print("Data loaded successfully!")
print(f"\nTotal Records: {df.count():,}")
print(f"Total Columns: {len(df.columns)}")

## 1.4 Schema Inspection

In [ ]:
# Display schema
print("Dataset Schema:")
print("=" * 60)
df.printSchema()

In [ ]:
# Display column names and data types
print(f"{'Column Name':<30} {'Data Type':<20}")
print("=" * 50)
for field in df.schema.fields:
    print(f"{field.name:<30} {str(field.dataType):<20}")
print(f"\nTotal columns: {len(df.columns)}")

In [ ]:
# Show first 5 rows
print("Sample Data (First 5 rows):")
df.show(5, truncate=False)

## 1.5 Big Data Verification
Verify the dataset meets the Big Data requirements (>= 1GB, > 10 columns).

In [ ]:
# Big Data Requirements Verification
num_records = df.count()
num_columns = len(df.columns)

# Calculate dataset size in memory
# Convert to pandas sample to estimate row size
sample_pd = df.limit(1000).toPandas()
avg_row_size = sample_pd.memory_usage(deep=True).sum() / len(sample_pd)
estimated_size_gb = (avg_row_size * num_records) / (1024**3)

# Calculate parquet file sizes on disk
import glob
data_dir = r"C:\Users\MaK Tech\Desktop\nyc-taxi-bigdata\data"
parquet_files = glob.glob(os.path.join(data_dir, "*.parquet"))
total_disk_size = sum(os.path.getsize(f) for f in parquet_files)
disk_size_gb = total_disk_size / (1024**3)

print("=" * 60)
print("BIG DATA REQUIREMENTS VERIFICATION")
print("=" * 60)
print(f"\nTotal Records:          {num_records:,}")
print(f"Total Columns:          {num_columns}")
print(f"Parquet Size on Disk:   {disk_size_gb:.2f} GB")
print(f"Estimated In-Memory:    {estimated_size_gb:.2f} GB")
print(f"Number of Files:        {len(parquet_files)}")
print(f"\n{'Requirement':<30} {'Status':<10} {'Value'}")
print("-" * 60)
print(f"{'Size >= 1GB':<30} {'PASS' if estimated_size_gb >= 1 else 'CHECK':<10} {estimated_size_gb:.2f} GB")
print(f"{'Columns > 10':<30} {'PASS' if num_columns > 10 else 'FAIL':<10} {num_columns} columns")
print(f"{'Not from Kaggle':<30} {'PASS':<10} AWS Open Data Registry")
print(f"{'Not lab dataset':<30} {'PASS':<10} NYC TLC Original Data")

## 1.6 Data Quality Assessment

In [ ]:
# Missing values analysis
print("Missing Values Analysis:")
print("=" * 60)

missing_counts = []
for col_name in df.columns:
    null_count = df.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / num_records) * 100
    missing_counts.append((col_name, null_count, null_pct))

print(f"{'Column':<30} {'Missing Count':<15} {'Missing %':<10}")
print("-" * 55)
for col_name, count, pct in missing_counts:
    print(f"{col_name:<30} {count:<15,} {pct:.2f}%")

In [ ]:
# Descriptive statistics for numerical columns
print("Descriptive Statistics:")
print("=" * 60)
df.describe().show()

## 1.7 Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of key numerical features
# Sample data for visualization (plotting all millions of rows is impractical)
sample_df = df.sample(fraction=0.01, seed=42).toPandas()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution of Key Features', fontsize=16, fontweight='bold')

# Trip distance
axes[0, 0].hist(sample_df['trip_distance'].clip(0, 30), bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Trip Distance (miles)')
axes[0, 0].set_xlabel('Distance')
axes[0, 0].set_ylabel('Frequency')

# Fare amount
axes[0, 1].hist(sample_df['fare_amount'].clip(0, 100), bins=50, color='coral', edgecolor='black')
axes[0, 1].set_title('Fare Amount ($)')
axes[0, 1].set_xlabel('Fare ($)')
axes[0, 1].set_ylabel('Frequency')

# Tip amount
axes[0, 2].hist(sample_df['tip_amount'].clip(0, 30), bins=50, color='mediumseagreen', edgecolor='black')
axes[0, 2].set_title('Tip Amount ($)')
axes[0, 2].set_xlabel('Tip ($)')
axes[0, 2].set_ylabel('Frequency')

# Total amount
axes[1, 0].hist(sample_df['total_amount'].clip(0, 150), bins=50, color='mediumpurple', edgecolor='black')
axes[1, 0].set_title('Total Amount ($)')
axes[1, 0].set_xlabel('Total ($)')
axes[1, 0].set_ylabel('Frequency')

# Passenger count
passenger_counts = sample_df['passenger_count'].dropna().value_counts().sort_index()
axes[1, 1].bar(passenger_counts.index.astype(str), passenger_counts.values, color='goldenrod', edgecolor='black')
axes[1, 1].set_title('Passenger Count')
axes[1, 1].set_xlabel('Passengers')
axes[1, 1].set_ylabel('Frequency')

# Payment type
payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute', 5: 'Unknown'}
payment_counts = sample_df['payment_type'].map(payment_labels).value_counts()
axes[1, 2].bar(payment_counts.index, payment_counts.values, color='lightcoral', edgecolor='black')
axes[1, 2].set_title('Payment Type')
axes[1, 2].tick_params(axis='x', rotation=45)
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/samples/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to data/samples/feature_distributions.png")

In [ ]:
# Trip patterns by hour of day
df_with_hour = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
hourly_trips = (df_with_hour.groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
    .toPandas())

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hourly_trips['pickup_hour'], hourly_trips['count'], color='steelblue', edgecolor='black')
ax.set_title('Trip Volume by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Trips')
ax.set_xticks(range(24))
plt.tight_layout()
plt.savefig('../data/samples/hourly_trips.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to data/samples/hourly_trips.png")

In [ ]:
# Monthly trip volume
df_with_month = df.withColumn("pickup_month", F.month("tpep_pickup_datetime"))
monthly_trips = (df_with_month.groupBy("pickup_month")
    .count()
    .orderBy("pickup_month")
    .toPandas())

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(monthly_trips['pickup_month'], monthly_trips['count'], 
              color='coral', edgecolor='black')
ax.set_title('Trip Volume by Month (2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Trips')
ax.set_xticks(monthly_trips['pickup_month'])
ax.set_xticklabels([month_names[m-1] for m in monthly_trips['pickup_month']])
plt.tight_layout()
plt.savefig('../data/samples/monthly_trips.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to data/samples/monthly_trips.png")

In [ ]:
# Correlation matrix of numerical features
numerical_cols = ['trip_distance', 'fare_amount', 'tip_amount', 'tolls_amount',
                  'total_amount', 'passenger_count', 'congestion_surcharge']

corr_df = sample_df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/samples/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to data/samples/correlation_matrix.png")

## 1.8 Data Partitioning & Storage
Save cleaned data in optimized Parquet format with partitioning for efficient downstream processing.

In [ ]:
# Cache the dataframe for reuse
df.persist()
print(f"DataFrame cached with {df.rdd.getNumPartitions()} partitions")

# Repartition by month for efficient query patterns
df_partitioned = (df
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .repartition(12, "pickup_month"))

# Save as optimized Parquet
output_path = r"C:\Users\MaK Tech\Desktop\nyc-taxi-bigdata\data\processed"
df_partitioned.write.mode("overwrite").partitionBy("pickup_month").parquet(output_path)
print(f"\nPartitioned data saved to: {output_path}")
print("Storage format: Parquet (columnar, compressed)")
print("Partitioning strategy: By pickup month (aligned with temporal query patterns)")

## 1.9 Summary

### Key Findings:
- Dataset contains millions of taxi trip records from 2024
- 18+ features covering trip details, fares, and payment information
- Dataset size exceeds 1GB requirement for Big Data classification
- Peak trip hours are during evening rush (5-7 PM)
- Strong correlation between trip distance and fare amount
- Credit card is the dominant payment method

### Big Data Challenges Identified:
- Memory management needed for processing millions of records
- Efficient partitioning strategy required for temporal queries
- Parquet format chosen for columnar compression and fast I/O

### Next Steps:
- Feature engineering (Notebook 2)
- Model training with PySpark MLlib (Notebook 3)
- Model evaluation and comparison (Notebook 4)

In [ ]:
# Unpersist cached data and stop Spark
df.unpersist()
spark.stop()
print("Spark session stopped. Notebook 1 complete.")